In [1]:
import pickle  
import firebase_admin
import pandas as pd
import numpy as np
from firebase_admin import db
from firebase_admin import credentials
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore", message="X has feature names, but RandomForestClassifier was fitted without feature names")

In [2]:
live_data = 'liveData'

cred = credentials.Certificate("greenhouse2024-3534b-firebase-adminsdk-2en30-352a40c0af.json")
default_app = firebase_admin.initialize_app(cred, {
                                            'databaseURL':'https://greenhouse2024-3534b-default-rtdb.firebaseio.com/'
                                            })

ref_live = db.reference(live_data)


In [3]:
# Load model
with open('rgb_model.pkl', 'rb') as file:
    model = pickle.load(file)

print(f"Loaded model type: {type(model)}")  # Debugging step

Loaded model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [4]:
def inference_func_tabular(sample_json):
    df = pd.DataFrame([sample_json])  # Convert dict to DataFrame
    predicted_rgb = model.predict(df)
    return predicted_rgb.tolist()[0]  # Convert NumPy array to list

In [5]:
def firebase_pipeline_tabular():
    all_data = ref_live.get()
    keys_to_delete = []
    
    if not all_data:
        return  # Avoid errors if no data in Firebase
    
    for data, value in all_data.items():
        if data == 'isNew' and value == "true":
            keys_to_delete.append('isNew')

    for data in keys_to_delete:
        del all_data[data]

        data_dup = all_data.copy()
        
        # Extracting values safely
        temperature = int(data_dup.get('temperature', 0))
        humidity = int(data_dup.get('humidity', 0))
        lux = int(data_dup.get('lux', 0))
        startdate = int(data_dup.get('date', 0))

        timestamp_s = startdate / 1000  # Convert milliseconds to seconds
        target_date = datetime.utcfromtimestamp(timestamp_s)
        today = datetime.utcnow()

        delta_weeks = (today - target_date).days / 7
        if delta_weeks < 1:
            delta_weeks = 1

        sample_data = {
            "temperature": temperature,
            "humidity": humidity,
            "lux_level": lux,
            "week": int(delta_weeks)
        }

        print(sample_data)
        ref_live.update({'isNew': "false"})

        try:
            status = inference_func_tabular(sample_data)
            print("Prediction:", status)
            prediction_int_str = f"{int(status[0])},{int(status[1])},{int(status[2])}"
            ref_live.update({'Prediction': prediction_int_str}) 

        except Exception as e:
            print("Prediction error:", e)

In [6]:
while True:
    firebase_pipeline_tabular()

{'temperature': 26, 'humidity': 61, 'lux_level': 8, 'week': 2890}
Prediction: [207.26, 177.22, 7.83]
{'temperature': 26, 'humidity': 61, 'lux_level': 12, 'week': 2890}
Prediction: [207.26, 177.22, 7.83]
{'temperature': 26, 'humidity': 61, 'lux_level': 20, 'week': 2890}
Prediction: [207.26, 177.22, 7.83]
{'temperature': 26, 'humidity': 61, 'lux_level': 20, 'week': 2890}
Prediction: [207.26, 177.22, 7.83]


UnavailableError: Failed to establish a connection: HTTPSConnectionPool(host='greenhouse2024-3534b-default-rtdb.firebaseio.com', port=443): Max retries exceeded with url: /liveData.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001EF28D677C0>: Failed to resolve 'greenhouse2024-3534b-default-rtdb.firebaseio.com' ([Errno 11001] getaddrinfo failed)"))